## Data Handling and Memory Management

In [1]:
import pandas as pd
import numpy as np
import os
import glob
import gc
from tqdm import tqdm
import psutil
import warnings
warnings.filterwarnings('ignore')




In [2]:
def get_ram_usage():
    process = psutil.Process(os.getpid())
    ram_mb = process.memory_info().rss / 1024 / 1024
    return ram_mb
print(f"Current RAM usage: {get_ram_usage():.1f} MB")
print(f"Total system RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB")
print(f"Available RAM: {psutil.virtual_memory().available / 1024**3:.1f} GB")

Current RAM usage: 97.0 MB
Total system RAM: 7.7 GB
Available RAM: 1.5 GB


## Exploring the Dataset

In [4]:
DATA_PATH = r"C:\Users\LENOVO\Downloads\milan_data"
all_files = sorted(glob.glob(os.path.join(DATA_PATH, "sms-call-internet-mi-*.txt")))
print(f"{len(all_files)} files")
total_size_gb = sum(os.path.getsize(f) for f in all_files) / 1024**3
print(f" Total raw data size on disk: {total_size_gb:.2f} GB")
print("\nFirst 3 files:")
for f in all_files[:3]:
    print(f"  {os.path.basename(f)}")
print("Last 3 files:")
for f in all_files[-3:]:
    print(f"  {os.path.basename(f)}")

62 files
 Total raw data size on disk: 19.38 GB

First 3 files:
  sms-call-internet-mi-2013-11-01.txt
  sms-call-internet-mi-2013-11-02.txt
  sms-call-internet-mi-2013-11-03.txt
Last 3 files:
  sms-call-internet-mi-2013-12-30.txt
  sms-call-internet-mi-2013-12-31.txt
  sms-call-internet-mi-2014-01-01.txt


In [3]:
sample = pd.read_csv(all_files[0], sep='\t', header=None, nrows=5)
print("Raw file preview:")
print(sample)
print(f"\nNumber of columns: {len(sample.columns)}")
print(f"\nData types:\n{sample.dtypes}")

Raw file preview:
   0              1   2         3         4         5         6          7
0  1  1383260400000   0  0.081363       NaN       NaN       NaN        NaN
1  1  1383260400000  39  0.141864  0.156787  0.160938  0.052275  11.028366
2  1  1383261000000   0  0.136588       NaN       NaN  0.027300        NaN
3  1  1383261000000  33       NaN       NaN       NaN       NaN   0.026137
4  1  1383261000000  39  0.278452  0.119926  0.188777  0.133637  11.100963

Number of columns: 8

Data types:
0      int64
1      int64
2      int64
3    float64
4    float64
5    float64
6    float64
7    float64
dtype: object


The dataset consists of 62 daily text files covering the period from November 1, 2013 to January 1, 2014. Each file follows a tab-separated format with 8 columns

Observations : Many rows has NaN values in the internet traffic column, meaning those records belonged to SMS or call activity only, with no internet usage recorded. Second, the default data types assigned by pandas were all 64-bit, which is unnecessarily large for our purposes.

To understand the scale of the challenge, we loaded one file without any optimization

In [6]:
df_unoptimized = pd.read_csv(all_files[0], sep='\t', header=None)

mem_one_file = df_unoptimized.memory_usage(deep=True).sum() / 1024**2

print(f"\nBEFORE OPTIMIZATION (1 file)")
print(f"Rows in one file: {len(df_unoptimized):,}")
print(f"RAM used by one file: {mem_one_file:.1f} MB")
print(f"All columns: {list(df_unoptimized.columns)}")
print(f"Data types:\n{df_unoptimized.dtypes}")
print(f"\nEstimated for all 62 files (no optimization): {mem_one_file * 62 / 1024:.2f} GB")


del df_unoptimized
gc.collect()


BEFORE OPTIMIZATION (1 file)
Rows in one file: 4,842,625
RAM used by one file: 295.6 MB
All columns: [0, 1, 2, 3, 4, 5, 6, 7]
Data types:
0      int64
1      int64
2      int64
3    float64
4    float64
5    float64
6    float64
7    float64
dtype: object

Estimated for all 62 files (no optimization): 17.90 GB


0

In [7]:
def load_file_optimized(filepath):
    df = pd.read_csv(
        filepath, 
        sep='\t', 
        header=None,
        usecols=[0, 1, 7], 
        names=['square_id', 'time_interval', 'internet'],
        dtype={
            0: np.int16,    
            1: np.int64,    
            7: np.float32   
        }
    )
    
    df = df.dropna(subset=['internet'])
    
    return df

print("Testing optimized loader on one file...")
df_test = load_file_optimized(all_files[0])

mem_optimized = df_test.memory_usage(deep=True).sum() / 1024**2

print(f"Rows after dropping NaN: {len(df_test):,}")
print(f"RAM used: {mem_optimized:.1f} MB")
print(f"Data types:\n{df_test.dtypes}")
print(f"Sample data:\n{df_test.head()}")
print(f"\nEstimated for all 62 files: {mem_optimized * 62 / 1024:.2f} GB")
print(f"\nMemory reduced:  {mem_optimized:.1f} MB")
del df_test
gc.collect()

Testing optimized loader on one file...
Rows after dropping NaN: 2,353,999
RAM used: 49.4 MB
Data types:
square_id          int16
time_interval      int64
internet         float32
dtype: object
Sample data:
   square_id  time_interval   internet
1          1  1383260400000  11.028366
3          1  1383261000000   0.026137
4          1  1383261000000  11.100964
6          1  1383261600000  10.892771
8          1  1383262200000   8.622424

Estimated for all 62 files: 2.99 GB

Memory reduced:  49.4 MB


0

Memory Optimization

I applied four techniques to bring memory usage to a manageable level
Column Selection :Instead of loading all 8 columns, we loaded only the 3 relevant ones using the usecols parameter
Data Type Downcasting :The default pandas behavior assigns 64-bit types to all numeric columns. We replaced these with smaller types where appropriate.
Dropping NaN Rows: Rows where internet traffic was NaN represented SMS/call-only records with no value for our analysis.
File-by-File Loading with Immediate Concatenation :Instead of holding all 62 dataframes simultaneously, we loaded one file at a time, appended it to a list, and then concatenated once.
I was able to reduce 83% reduction per file.

In [13]:
import os

PARQUET_PATH = os.path.join(DATA_PATH, 'traffic_data.parquet')

if os.path.exists(PARQUET_PATH):
    print("Loading saved data")
    df_all = pd.read_parquet(PARQUET_PATH)
    print(f"Loaded {len(df_all):,} rows")

else:
    
    ram_start = get_ram_usage()
    frames = []
    
    for i, filepath in enumerate(tqdm(all_files, desc="Loading files")):
        df_temp = load_file_optimized(filepath)
        frames.append(df_temp)
        del df_temp
    
    print("\nCombining all files")
    df_all = pd.concat(frames, ignore_index=True)
    del frames
    gc.collect()
    
    print(" Converting timestamps...")
    df_all['time_interval'] = pd.to_datetime(df_all['time_interval'], unit='ms')

    print(" Removing duplicates")
    rows_before = len(df_all)
    df_all = df_all.drop_duplicates()
    rows_after = len(df_all)
    print(f"Removed {rows_before - rows_after:,} duplicate rows")
    
    df_all = df_all.sort_values(['square_id', 'time_interval']).reset_index(drop=True)
    
    print(" Saving to parquet")
    df_all.to_parquet(PARQUET_PATH, index=False)
    
    ram_end = get_ram_usage()
    mem_final = df_all.memory_usage(deep=True).sum() / 1024**2

    print(f"Total rows: {len(df_all):,}")
    print(f"RAM used: {mem_final:.1f} MB")
    print(f"Date range: {df_all['time_interval'].min()} → {df_all['time_interval'].max()}")
    print(f"Unique areas: {df_all['square_id'].nunique():,}")
    print(f"Saved to: {PARQUET_PATH}")

Loading saved data
Loaded 150,125,712 rows


Data Cleaning 

I had to Deduplication to identify and remove rows that are identical across all columns.
After cleaning, the data was transformed from a long format into a wide matrix format.

In [15]:
print(f"{'Raw files on disk':<35} {'19.38 GB':>18}")
print(f"{'Files processed':<35} {'62':>18}")
print(f"{'Total rows (before dedup)':<35} {'160,681,352':>18}")
print(f"{'Duplicate rows removed':<35} {'10,555,640':>18}")
print(f"{'Clean rows remaining':<35} {'150,125,712':>18}")
print(f"{'Unique grid areas':<35} {'10,000':>18}")
print(f"{'Date range start':<35} {'2013-10-31':>18}")
print(f"{'Date range end':<35} {'2014-01-01':>18}")
print(f"{'Memory per file (unoptimized)':<35} {'295.6 MB':>18}")
print(f"{'Memory per file (optimized)':<35} {'49.4 MB':>18}")
print(f"{'Memory reduction per file':<35} {'83.3%':>18}")
print(f"{'Est. total unoptimized':<35} {'17.90 GB':>18}")
print(f"{'Actual total in RAM':<35} {'2.00 GB':>18}")
print(f"{'Columns dropped':<35} {'5 of 8':>18}")

Raw files on disk                             19.38 GB
Files processed                                     62
Total rows (before dedup)                  160,681,352
Duplicate rows removed                      10,555,640
Clean rows remaining                       150,125,712
Unique grid areas                               10,000
Date range start                            2013-10-31
Date range end                              2014-01-01
Memory per file (unoptimized)                 295.6 MB
Memory per file (optimized)                    49.4 MB
Memory reduction per file                        83.3%
Est. total unoptimized                        17.90 GB
Actual total in RAM                            2.00 GB
Columns dropped                                 5 of 8
